In [ ]:
# ============================================================
# STEERING INFERENCE GRID ALPHA — CLASSIFIER ROUTING
# ============================================================

import re
import random
import subprocess
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


# ============================================================
# PATHS 
# ============================================================

# ROOT_DIR is a root directory of the project. Put the path instead of "...".
ROOT_DIR = Path(r"...")

MODEL_DIR = ROOT_DIR / "models" / "MODEL" # choose the model from the models directory
TEST_PATH = ROOT_DIR / "data" / "datasets" / "test_all.xlsx"

OUT_PATH = ROOT_DIR / "RESULT.xlsx" #you can change the name and path of the output file or keep the default one
AGG_OUT_PATH = OUT_PATH.with_name(OUT_PATH.stem + "_aggregated.xlsx")

CLASSIFIER_PATH = ROOT_DIR / "classifier" / "pytorch_equation_classifier.pt"

STEERING_PATHS = { 
    "polynomial": ROOT_DIR / "steering" / "polynomial" / "steering_polynomial_short_qwen.pt", #choose steering vector file for the chosen model
    "separable": ROOT_DIR / "steering" / "separable" / "steering_separable_short_qwen.pt", #choose steering vector file for the chosen model
    "unhomogenous": ROOT_DIR / "steering" / "inhomogeneous" / "steering_inhomogeneous_short_qwen.pt", #choose steering vector file for the chosen model
}


# ============================================================
# CONFIG — CUDA only
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this notebook. No CPU fallback is allowed.")

DEVICE = torch.device("cuda")
TORCH_DTYPE = torch.float16
GEN_MAX_NEW_TOKENS = 2000 #max number of tokens to generate. 500 and 2000 were tested.

N_RUNS = 1

# ===== ALPHA GRID =====
# ALPHA_GRID = [1.0]
ALPHA_GRID = [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5]

# ============================================================
# GPU INFO
# ============================================================

def print_gpu_info():
    print("\n" + "=" * 80)
    print("CUDA / GPU INFO")
    print("=" * 80)

    print(f"torch version       : {torch.__version__}")
    print(f"torch CUDA version  : {torch.version.cuda}")
    print(f"CUDA available      : {torch.cuda.is_available()}")
    print(f"CUDA device count   : {torch.cuda.device_count()}")
    print(f"current CUDA device : {torch.cuda.current_device()}")

    for device_id in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(device_id)
        total_gb = props.total_memory / 1024**3

        print("\n" + "-" * 80)
        print(f"GPU {device_id}")
        print("-" * 80)
        print(f"name                : {props.name}")
        print(f"compute capability  : {props.major}.{props.minor}")
        print(f"total VRAM          : {total_gb:.2f} GB")
        print(f"multiprocessors     : {props.multi_processor_count}")

    active_id = torch.cuda.current_device()
    allocated_gb = torch.cuda.memory_allocated(active_id) / 1024**3
    reserved_gb = torch.cuda.memory_reserved(active_id) / 1024**3

    print("\n" + "-" * 80)
    print(f"ACTIVE GPU MEMORY BEFORE MODEL LOAD: cuda:{active_id}")
    print("-" * 80)
    print(f"allocated           : {allocated_gb:.3f} GB")
    print(f"reserved            : {reserved_gb:.3f} GB")

    try:
        smi = subprocess.run(
            ["nvidia-smi"],
            capture_output=True,
            text=True,
            check=False,
        )
        if smi.returncode == 0:
            print("\n" + "-" * 80)
            print("nvidia-smi")
            print("-" * 80)
            print(smi.stdout)
        else:
            print("\n nvidia-smi is not available or returned an error.")
    except Exception as e:
        print(f"\n nvidia-smi check skipped: {type(e).__name__}: {e}")

    print("=" * 80 + "\n")


print_gpu_info()


def print_cuda_memory(label: str):
    active_id = torch.cuda.current_device()
    allocated_gb = torch.cuda.memory_allocated(active_id) / 1024**3
    reserved_gb = torch.cuda.memory_reserved(active_id) / 1024**3

    print("\n" + "=" * 80)
    print(f"CUDA MEMORY — {label}: cuda:{active_id}")
    print("=" * 80)
    print(f"allocated           : {allocated_gb:.3f} GB")
    print(f"reserved            : {reserved_gb:.3f} GB")
    print("=" * 80 + "\n")




# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# ============================================================
# UTILS: robust nested \boxed{...} extraction
# ============================================================

def extract_boxed(text: str) -> str:
    if text is None:
        return ""

    text = str(text)
    matches = [m.start() for m in re.finditer(r"\\boxed\{", text)]
    if not matches:
        return ""

    start = matches[-1] + len(r"\boxed{")
    depth, i = 1, start

    while i < len(text) and depth:
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
        i += 1

    if depth != 0:
        return ""

    return text[start:i - 1].strip()


def strip_boxed_if_present(text: str) -> str:
    text = "" if text is None else str(text)
    boxed = extract_boxed(text)
    return boxed if boxed else text


# ============================================================
# BLEU + NORMALIZATION
# ============================================================

def normalize_answer(s: str, predicted_class: str) -> str:
    if not s:
        return ""

    s = strip_boxed_if_present(str(s))

    if predicted_class == 'polynomial':
        # remove y= part
        s = re.sub(r"^y\s*=\s*", "", s)
        s = s.replace("\\left", "").replace("\\right", "")
        s = re.sub(r"\s+", "", s)

    elif predicted_class == 'separable':
        s = re.sub(r"^y\s*=\s*", "", s)
        s = s.replace("\\left", "").replace("\\right", "")
        s = s.replace(" ", "")

    elif predicted_class == 'unhomogenous':
        if "=" in s:
            s = s.split("=", 1)[-1]

        s = s.replace("\\left", "").replace("\\right", "")
        s = re.sub(r"\\frac\{([^{}]+)\}\{([^{}]+)\}", r"(\1)/(\2)", s)
        s = re.sub(r"\\sqrt\{([^{}]+)\}", r"sqrt(\1)", s)
        s = re.sub(r"e\^\{([^{}]+)\}", r"exp(\1)", s)
        s = re.sub(r"\s+", "", s)

    else:
        s = s.replace("\\left", "").replace("\\right", "")
        s = re.sub(r"\s+", "", s)

    return s


def post_tokenize_math(expr: str, predicted_class: str):
    if predicted_class == 'polynomial':
        return re.findall(r"[A-Za-z]+|\d+|\^|\+|\-|\*|\/|\(|\)|C", expr)
    if predicted_class == 'separable':
        return re.findall(r"[A-Za-z]+|\d+|\^|\+|\-|\*|\/|\(|\)|\{|\}|C", expr)
    if predicted_class == 'unhomogenous':
        return re.findall(r"[A-Za-z]+|\d+|\+|\-|\*|\/|\(|\)", expr)
    return re.findall(r"[A-Za-z]+|\d+|\^|\+|\-|\*|\/|\(|\)|\{|\}|C", expr)


def compute_bleu(true: str, pred: str, predicted_class: str) -> float:
    if not true or not pred:
        return 0.0

    true_tokens = post_tokenize_math(true, predicted_class)
    pred_tokens = post_tokenize_math(pred, predicted_class)

    if not true_tokens or not pred_tokens:
        return 0.0

    return sentence_bleu(
        [true_tokens],
        pred_tokens,
        weights=(0.5, 0.5),
        smoothing_function=SmoothingFunction().method1,
    )


def normalize_equation(s: str) -> str:
    if not s:
        return ""

    s = strip_boxed_if_present(str(s))
    s = s.replace("\\left", "").replace("\\right", "")
    s = re.sub(r"\\frac\{([^{}]+)\}\{([^{}]+)\}", r"(\1)/(\2)", s)
    s = re.sub(r"\\sqrt\{([^{}]+)\}", r"sqrt(\1)", s)
    s = re.sub(r"e\^\{([^{}]+)\}", r"exp(\1)", s)
    s = re.sub(r"\s+", "", s)

    return s


def pre_tokenize_math(expr: str):
    return re.findall(r"[A-Za-z]+|\d+|\+|\-|\*|\/|\(|\)|=|\^|\{|\}|_|'", expr)


def preprocess_equation(equation: str) -> str:
    normalized = normalize_equation(equation)
    tokens = pre_tokenize_math(normalized)
    return " ".join(tokens)


# ============================================================
# CLASSIFIER
# ============================================================

class MathTextCNN(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        embed_dim: int = 96,
        num_filters: int = 128,
        kernel_sizes: tuple[int, ...] = (3, 5, 7),
        dropout: float = 0.25,
        padding_idx: int = 0,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.convs = nn.ModuleList(
            [
                nn.Conv1d(embed_dim, num_filters, kernel_size=k, padding=k // 2)
                for k in kernel_sizes
            ]
        )
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, x):
        embedded = self.embedding(x).transpose(1, 2)

        pooled = []
        for conv in self.convs:
            features = self.activation(conv(embedded))
            pooled.append(torch.amax(features, dim=-1))

        features = torch.cat(pooled, dim=1)
        features = self.dropout(features)

        return self.classifier(features)


class EquationTypeClassifier:
    def __init__(self, model_path: str | Path, device: torch.device):
        self.device = device

        checkpoint = torch.load(model_path, map_location=self.device, weights_only=False)

        self.vocab = checkpoint["vocab"]
        self.max_len = checkpoint["max_len"]
        self.label_names = checkpoint["label_names"]

        config = checkpoint.get("config", {})

        self.model = MathTextCNN(
            vocab_size=len(self.vocab),
            num_classes=len(self.label_names),
            embed_dim=config.get("embed_dim", 96),
            num_filters=config.get("num_filters", 128),
            dropout=config.get("dropout", 0.25),
        )

        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.model.to(self.device)
        self.model.eval()

    def encode(self, equation: str) -> torch.Tensor:
        token_text = preprocess_equation(equation)
        tokens = token_text.split()

        ids = [self.vocab.get(token, self.vocab["<UNK>"]) for token in tokens]
        ids = ids[: self.max_len]

        if len(ids) < self.max_len:
            ids += [self.vocab["<PAD>"]] * (self.max_len - len(ids))

        return torch.tensor([ids], dtype=torch.long, device=self.device)

    def predict(self, equation: str) -> str:
        x = self.encode(equation)

        with torch.no_grad():
            logits = self.model(x)
            predicted_id = int(torch.argmax(logits, dim=1).item())

        return self.label_names[predicted_id]


def classify_equation(equation: str) -> str:
    return classifier.predict(equation)


# ------------------------------------------------------------
# PROMPT
# ------------------------------------------------------------
BASE_SYS = """
You are a symbolic mathematics model.

Task: compute y(x) from the given derivative y'(x).

Output:
- ONLY final answer
- LaTeX
- \\boxed{y=...+C}
"""


def make_prompt(eq: str, predicted_class: str) -> str:
    if predicted_class == 'polynomial':
        return (
            "You are a symbolic mathematics model.\n\n"
            "Task: compute y(x) from the given derivative y'(x).\n\n"
            "Requirements:\n"
            "- Output ONLY the final explicit polynomial y(x).\n"
            "- Do NOT output integrals.\n"
            "- Do NOT output the symbol \\int.\n"
            "- Use LaTeX.\n"
            "- Return exactly one boxed expression of the form \\boxed{y=...+C}.\n"
            "- Include +C.\n"
            "- No reasoning.\n"
            "- No explanations.\n\n"
            f"PROBLEM:\n{eq}\n\n"
            "ANSWER:\n"
        )

    if predicted_class == 'separable':
        return BASE_SYS + f"\nPROBLEM:\n{eq}\n\nANSWER:\n"

    if predicted_class == 'unhomogenous':
        return (
            "You are a symbolic mathematics model.\n"
            "Solve the differential equation analytically.\n\n"
            "EQUATION TYPE:\n inhomogenous 2nd order.\n\n"
            f"PROBLEM (LaTeX):\n{eq}\n\n"
            "Return ONLY the final answer in LaTeX boxed form.\nFINAL:\n"
        )

    raise ValueError(f"Unknown predicted_class: {predicted_class}")


# ============================================================
# MODEL
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=TORCH_DTYPE,
    device_map={"": 0},
)

model.eval()
torch.set_grad_enabled(False)

print("\nMODEL DEVICE CHECK")
print("=" * 80)
print("first model parameter device:", next(model.parameters()).device)
print("model dtype:", next(model.parameters()).dtype)
print_cuda_memory("AFTER MODEL LOAD")



# ============================================================
# CLASSIFIER LOAD — CUDA
# ============================================================

classifier = EquationTypeClassifier(CLASSIFIER_PATH, device=DEVICE)
print("classifier device:", next(classifier.model.parameters()).device)
print_cuda_memory("AFTER CLASSIFIER LOAD")


# ============================================================
# STEERING
# ============================================================

class Steering(nn.Module):
    def __init__(self, model, alpha: float):
        super().__init__()
        self.layers = model.model.layers
        h = model.config.hidden_size
        self.vectors = nn.Parameter(
            torch.zeros(len(self.layers), h, device=DEVICE, dtype=TORCH_DTYPE),
            requires_grad=False,
        )
        self.alpha = alpha
        self.handles = []

    def install(self):
        self.remove()

        def make_hook(i):
            def hook(_, __, out):
                return out + (self.alpha * self.vectors[i]).to(dtype=out.dtype, device=out.device)
            return hook

        for i, layer in enumerate(self.layers):
            self.handles.append(layer.mlp.down_proj.register_forward_hook(make_hook(i)))

    def remove(self):
        for h in self.handles:
            try:
                h.remove()
            except Exception:
                pass
        self.handles = []


steering = Steering(model, 1.0)
steering.install()


def load_steering_bank(paths: dict[str, Path]) -> dict[str, torch.Tensor]:
    bank = {}
    expected_shape = tuple(steering.vectors.shape)

    for label, path in paths.items():
        ckpt = torch.load(path, map_location=DEVICE)
        vectors = ckpt["vectors"].to(device=DEVICE, dtype=TORCH_DTYPE)

        if tuple(vectors.shape) != expected_shape:
            raise ValueError(
                f"Unexpected steering vector shape for {label}: "
                f"got {tuple(vectors.shape)}, expected {expected_shape}"
            )

        bank[label] = vectors
        print(f"Loaded steering vectors for {label}: {path}")

    return bank


STEERING_BANK = load_steering_bank(STEERING_PATHS)
print_cuda_memory("AFTER STEERING BANK LOAD")


def set_steering_vectors(predicted_class: str):
    if predicted_class not in STEERING_BANK:
        raise ValueError(f"No steering vectors for predicted_class={predicted_class}")

    with torch.no_grad():
        steering.vectors.copy_(STEERING_BANK[predicted_class])


# ============================================================
# GENERATE
# ============================================================

@torch.no_grad()
def generate_answer(eq: str, predicted_class: str, alpha: float) -> str:
    set_steering_vectors(predicted_class)
    steering.alpha = alpha

    prompt = make_prompt(eq, predicted_class)
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    seq = model.generate(
    **enc,
    max_new_tokens=GEN_MAX_NEW_TOKENS,
    do_sample=False,
    use_cache=True,
    pad_token_id=tokenizer.eos_token_id,)

    txt = tokenizer.decode(seq[0], skip_special_tokens=True)
    boxed = extract_boxed(txt)

    return boxed


# ============================================================
# TYPE_EQ CANONICALIZATION
# ============================================================

TYPE_EQ_TO_CANONICAL = {
    "polynomial": "polynomial",
    "polynomial equation": "polynomial",
    "polynomial equations": "polynomial",

    "separable": "separable",
    "separable variable": "separable",
    "separable variables": "separable",

    "unhomogenous": "unhomogenous",
    "unhomogeneous": "unhomogenous",
    "inhomogenous": "unhomogenous",
    "inhomogeneous": "unhomogenous",
    "unhomogenous 2nd order": "unhomogenous",
    "unhomogeneous 2nd order": "unhomogenous",
    "inhomogenous 2nd order": "unhomogenous",
    "inhomogeneous 2nd order": "unhomogenous",
}


def canonicalize_type_eq(type_eq: str) -> str:
    key = "" if type_eq is None else str(type_eq).strip().lower()
    return TYPE_EQ_TO_CANONICAL.get(key, key)


# ============================================================
# MAIN LOOP
# ============================================================

df = pd.read_excel(TEST_PATH)
print(f"Loaded dataset rows: {len(df)}")

# Shuffle test rows after reading the Excel file.
# random_state=SEED makes the permutation reproducible across runs.
df = df.dropna(subset=["true_answer", "equation"]).reset_index(drop=True)
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print(f"Dataset rows after dropna + shuffle: {len(df)} | shuffle_seed={SEED}")

all_rows = []

print(f"Dataset size: {len(df)}")

bleu_off_runs = []
bleu_on_runs = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    eq = str(row["equation"])
    type_eq_raw = str(row["type_eq"]).strip()
    type_eq = canonicalize_type_eq(type_eq_raw)
    true_ans = str(row["true_answer"])

    predicted_class = classify_equation(eq)
    classifier_correct = predicted_class == type_eq

    norm_true = normalize_answer(true_ans, predicted_class)

    for run in range(N_RUNS):
        
        pred_off = generate_answer(eq, predicted_class, alpha=0.0)
        norm_off = normalize_answer(pred_off, predicted_class)
        bleu_off = compute_bleu(norm_true, norm_off, predicted_class)
        bleu_off_runs.append(bleu_off)

        for alpha in ALPHA_GRID:
            
            if alpha == 0.0:
                pred_on = pred_off
                bleu_on = bleu_off
            else:
                pred_on = generate_answer(eq, predicted_class, alpha=alpha)
                norm_on = normalize_answer(pred_on, predicted_class)
                bleu_on = compute_bleu(norm_true, norm_on, predicted_class)

            bleu_on_runs.append(bleu_on)

            print("\n" + "=" * 80)
            print(f"[{i}] alpha={alpha} run={run + 1}")
            print("EQ:", eq)
            print("TRUE:", true_ans)
            print(f"true_class_raw: {type_eq_raw} | true_class: {type_eq} | predicted_class: {predicted_class} | classifier_correct: {classifier_correct}")
            print(f"OFF: {pred_off} | BLEU={bleu_off:.4f}")
            print(f"ON : {pred_on} | BLEU={bleu_on:.4f}")

            all_rows.append({
                "alpha": alpha,
                "eq_id": i,
                "run": run,
                "equation": eq,
                "type_eq_raw": type_eq_raw,
                "type_eq": type_eq,
                "predicted_class": predicted_class,
                "classifier_correct": classifier_correct,
                "true_answer": true_ans,
                "pred_off": pred_off,
                "pred_on": pred_on,
                "bleu_off": bleu_off,
                "bleu_on": bleu_on,
                "delta_bleu": bleu_on - bleu_off,
            })

    print(f"\nAVG OFF BLEU: {np.mean(bleu_off_runs):.4f}")
    print(f"AVG ON  BLEU: {np.mean(bleu_on_runs):.4f}")


# ============================================================
# SAVE EXCEL
# ============================================================

out_df = pd.DataFrame(all_rows)
out_df.to_excel(OUT_PATH, index=False)

print("\nSaved full results to:", OUT_PATH)


In [ ]:
# ============================================================
# AGGREGATION — type_eq n predicted_class
# ============================================================

if "OUT_PATH" not in globals() or "AGG_OUT_PATH" not in globals():
    ROOT_DIR = Path(r"...")
    OUT_PATH = ROOT_DIR / "classifier_steering_results_qwen.xlsx"
    AGG_OUT_PATH = OUT_PATH.with_name(OUT_PATH.stem + "_aggregated.xlsx")

df = pd.read_excel(OUT_PATH)


agg_by_predicted_class = (
    df.groupby(["alpha", "predicted_class"], as_index=False)
      .agg(
          bleu_on_mean=("bleu_on", "mean"),
          bleu_off_mean=("bleu_off", "mean"),
          delta_bleu_mean=("delta_bleu", "mean"),
          n=("bleu_on", "size"),
      )
)


agg_by_type_eq = (
    df.groupby(["alpha", "type_eq"], as_index=False)
      .agg(
          bleu_on_mean=("bleu_on", "mean"),
          bleu_off_mean=("bleu_off", "mean"),
          delta_bleu_mean=("delta_bleu", "mean"),
          classifier_accuracy=("classifier_correct", "mean"),
          n=("bleu_on", "size"),
      )
)


agg_by_type_and_predicted = (
    df.groupby(["alpha", "type_eq", "predicted_class", "classifier_correct"], as_index=False)
      .agg(
          bleu_on_mean=("bleu_on", "mean"),
          bleu_off_mean=("bleu_off", "mean"),
          delta_bleu_mean=("delta_bleu", "mean"),
          n=("bleu_on", "size"),
      )
)


baseline_by_type_and_predicted = (
    df.loc[df["alpha"] == 0.0]
      .groupby(["type_eq", "predicted_class", "classifier_correct"], as_index=False)
      .agg(
          bleu_baseline_mean=("bleu_off", "mean"),
          n=("bleu_off", "size"),
      )
)


classifier_quality = (
    df.drop_duplicates(subset=["eq_id", "run"])
      .groupby(["type_eq", "predicted_class", "classifier_correct"], as_index=False)
      .size()
      .rename(columns={"size": "n"})
)

classifier_accuracy_total = pd.DataFrame({
    "metric": ["classifier_accuracy_total"],
    "value": [df.drop_duplicates(subset=["eq_id", "run"])["classifier_correct"].mean()],
})

with pd.ExcelWriter(AGG_OUT_PATH) as writer:
    agg_by_predicted_class.to_excel(writer, sheet_name="by_predicted_class", index=False)
    agg_by_type_eq.to_excel(writer, sheet_name="by_type_eq", index=False)
    agg_by_type_and_predicted.to_excel(writer, sheet_name="by_type_and_predicted", index=False)
    baseline_by_type_and_predicted.to_excel(writer, sheet_name="baseline", index=False)
    classifier_quality.to_excel(writer, sheet_name="classifier_quality", index=False)
    classifier_accuracy_total.to_excel(writer, sheet_name="classifier_accuracy_total", index=False)

print("AGGREGATION BY PREDICTED_CLASS")
print(agg_by_predicted_class)

print("\nAGGREGATION BY TYPE_EQ")
print(agg_by_type_eq)

print("\nAGGREGATION BY TYPE_EQ AND PREDICTED_CLASS")
print(agg_by_type_and_predicted)

print("\nCLASSIFIER ACCURACY TOTAL")
print(classifier_accuracy_total)

print("\nSaved aggregated results to:", AGG_OUT_PATH)
